In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd

In [3]:
import sys
sys.path.append("..")
import pandas as pd
import numpy as np

df = pd.read_csv("../data/interim/cohort_labeled.csv")
print(df.shape)
print(df.kidney_marker.value_counts())

(1506, 32)
kidney_marker
0.0    855
1.0    651
Name: count, dtype: int64


In [4]:
# Candidate predictors: none of these appear in the outcome formula except age and sex
candidates = ["RIDAGEYR", "RIAGENDR", "INDFMPIR", "LBXGH", "BMXBMI", "BMXWAIST",
              "BPXOSY1", "BPXODI1", "SMQ020", "SMQ040", "DIQ050"]

# Percent missing per candidate, and mean of each by outcome
miss = df[candidates].isna().mean().mul(100).round(1).rename("pct_missing")
print(miss)
print(df.groupby("kidney_marker")[candidates].mean().round(2).T)

# Coded variables: look for 7 / 9 (refused / don't know)
for c in ["SMQ020", "SMQ040", "DIQ050"]:
    print(c, df[c].value_counts(dropna=False).to_dict())

RIDAGEYR     0.0
RIAGENDR     0.0
INDFMPIR    13.5
LBXGH        1.5
BMXBMI       2.3
BMXWAIST     6.9
BPXOSY1     10.2
BPXODI1     10.2
SMQ020       0.0
SMQ040      53.6
DIQ050      17.9
Name: pct_missing, dtype: float64
kidney_marker     0.0     1.0
RIDAGEYR        59.41   64.91
RIAGENDR         1.49    1.43
INDFMPIR         2.55    2.32
LBXGH            7.26    7.79
BMXBMI          32.86   32.86
BMXWAIST       109.70  110.64
BPXOSY1        127.78  137.34
BPXODI1         75.37   75.40
SMQ020           1.57    1.50
SMQ040           2.38    2.47
DIQ050           1.81    1.60
SMQ020 {2.0: 805, 1.0: 699, 9.0: 1, 7.0: 1}
SMQ040 {nan: 807, 3.0: 473, 1.0: 178, 2.0: 48}
DIQ050 {2.0: 872, 1.0: 363, nan: 270, 9.0: 1}


recode and derive features

Turn raw survey codes into clean, meaningful features while keeping skip-pattern missingness separate from real missingness.

In [5]:
# 1. Convert refused (7) and don't know (9) responses to missing
for c in ["SMQ020", "SMQ040", "DIQ050"]:
    df[c] = df[c].replace({7: np.nan, 9: np.nan})

# 2. Derive smoking status using the questionnaire skip pattern
df["smoking"] = np.select(
    [
        df.SMQ020 == 2,
        (df.SMQ020 == 1) & (df.SMQ040 == 3),
        (df.SMQ020 == 1) & df.SMQ040.isin([1, 2])
    ],
    ["never", "former", "current"],
    default="unknown"
)

# 3. Insulin use: preserve missing responses
df["insulin"] = np.nan
df.loc[df.DIQ050 == 1, "insulin"] = 1
df.loc[df.DIQ050 == 2, "insulin"] = 0

# 4. Sex as binary variable
df["female"] = (df.RIAGENDR == 2).astype(int)

# 5. Feature checks
print("Smoking distribution:")
print(df["smoking"].value_counts(dropna=False))

print("\nInsulin distribution:")
print(df["insulin"].value_counts(dropna=False))

print("\nSex distribution:")
print(df["female"].value_counts(dropna=False))

print("\nSmoking by kidney marker:")
print(
    pd.crosstab(
        df.smoking,
        df.kidney_marker,
        normalize="index"
    ).round(2)
)

# 6. Candidate predictor missingness
candidates = [
    "RIDAGEYR", "RIAGENDR", "INDFMPIR", "LBXGH",
    "BMXBMI", "BMXWAIST", "BPXOSY1", "BPXODI1",
    "SMQ020", "SMQ040", "DIQ050"
]

print("\nMissingness per predictor (%):")

miss = (
    df[candidates]
    .isna()
    .mean()
    .mul(100)
    .round(1)
    .rename("pct_missing")
)

print(miss)

print("\nMean predictor values by kidney marker:")
print(
    df.groupby("kidney_marker")[candidates]
    .mean()
    .round(2)
    .T
)

# 7. Questionnaire response counts
print("\nQuestionnaire response counts:")

for c in ["SMQ020", "SMQ040", "DIQ050"]:
    print(c, df[c].value_counts(dropna=False).to_dict())

# 8. Save feature-engineered dataset
df.to_csv("../data/interim/cohort_features.csv", index=False)

print("\nSaved cohort_features.csv:", df.shape)

Smoking distribution:
smoking
never      805
former     473
current    226
unknown      2
Name: count, dtype: int64

Insulin distribution:
insulin
0.0    872
1.0    363
NaN    271
Name: count, dtype: int64

Sex distribution:
female
0    803
1    703
Name: count, dtype: int64

Smoking by kidney marker:
kidney_marker   0.0   1.0
smoking                  
current        0.56  0.44
former         0.52  0.48
never          0.60  0.40
unknown        0.50  0.50

Missingness per predictor (%):
RIDAGEYR     0.0
RIAGENDR     0.0
INDFMPIR    13.5
LBXGH        1.5
BMXBMI       2.3
BMXWAIST     6.9
BPXOSY1     10.2
BPXODI1     10.2
SMQ020       0.1
SMQ040      53.6
DIQ050      18.0
Name: pct_missing, dtype: float64

Mean predictor values by kidney marker:
kidney_marker     0.0     1.0
RIDAGEYR        59.41   64.91
RIAGENDR         1.49    1.43
INDFMPIR         2.55    2.32
LBXGH            7.26    7.79
BMXBMI          32.86   32.86
BMXWAIST       109.70  110.64
BPXOSY1        127.78  137.34
BPXODI1

In [6]:
from sklearn.model_selection import train_test_split
from pathlib import Path

# 1. Stratified train-test split (80:20)
train, test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["kidney_marker"],
    random_state=42
)

# 2. Check dataset shapes
print("Training set:", train.shape)
print("Testing set:", test.shape)

# 3. Check outcome distribution
print("\nKidney marker distribution:")
print("Training:")
print(train["kidney_marker"].value_counts())
print("\nTesting:")
print(test["kidney_marker"].value_counts())

# 4. Compare positive outcome proportions
print("\nPositive outcome proportions:")
print("Training:", round(train["kidney_marker"].mean(), 3))
print("Testing:", round(test["kidney_marker"].mean(), 3))

# 5. Check participant overlap
overlap = set(train["SEQN"]) & set(test["SEQN"])
print("\nSEQN overlap:", len(overlap))

assert len(overlap) == 0, "Participant leakage detected!"

# 6. Save train and test datasets
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

train.to_csv(output_dir / "train.csv", index=False)
test.to_csv(output_dir / "test.csv", index=False)

print("\nTrain and test datasets saved successfully.")

Training set: (1204, 35)
Testing set: (302, 35)

Kidney marker distribution:
Training:
kidney_marker
0.0    684
1.0    520
Name: count, dtype: int64

Testing:
kidney_marker
0.0    171
1.0    131
Name: count, dtype: int64

Positive outcome proportions:
Training: 0.432
Testing: 0.434

SEQN overlap: 0

Train and test datasets saved successfully.
